# 04 KV Cache Lab

用最小张量演示：过去的 K/V 只计算一次，新 token 只追加新的 K/V。

In [1]:
import torch

B, H, past_T, new_T, D = 1, 2, 4, 1, 8
K_past = torch.randn(B, H, past_T, D)
V_past = torch.randn(B, H, past_T, D)
K_new = torch.randn(B, H, new_T, D)
V_new = torch.randn(B, H, new_T, D)
Q_new = torch.randn(B, H, new_T, D)

K_total = torch.cat([K_past, K_new], dim=-2)
V_total = torch.cat([V_past, V_new], dim=-2)

print("K_past:", K_past.shape)
print("K_new:", K_new.shape)
print("K_total:", K_total.shape)

K_past: torch.Size([1, 2, 4, 8])
K_new: torch.Size([1, 2, 1, 8])
K_total: torch.Size([1, 2, 5, 8])


In [2]:
import math
weights = torch.softmax(
    Q_new @ K_total.transpose(-2, -1) / math.sqrt(D),
    dim=-1,
)
output_new = weights @ V_total
print("new query output:", output_new.shape)
print("attention over keys:", weights.shape)

new query output: torch.Size([1, 2, 1, 8])
attention over keys: torch.Size([1, 2, 1, 5])


## 为什么不需要过去的 Q？

生成当前 token 时只需要计算“当前新 Query 应该读取哪些历史信息”。过去位置的 Query 已经完成过它们自己的输出计算，因此不需要再缓存。

## Cache 的代价

KV Cache 减少重复计算，但尺寸大致随：

```text
num_layers × batch × heads × seq_len × head_dim × 2(K/V)
```

线性增长，所以长上下文生成会产生显著 Cache 显存。